## Cleaning Up the DataSet

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('DataSet.csv')
df = df.drop_duplicates()
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Movie Name          1698 non-null   str  
 1   Release Period      1698 non-null   str  
 2   Whether Remake      1698 non-null   str  
 3   Whether Franchise   1698 non-null   str  
 4   Genre               1698 non-null   str  
 5   New Actor           1698 non-null   str  
 6   New Director        1698 non-null   str  
 7   New Music Director  1698 non-null   str  
 8   Lead Star           1698 non-null   str  
 9   Director            1698 non-null   str  
 10  Music Director      1698 non-null   str  
 11  Number of Screens   1698 non-null   int64
 12  Revenue(INR)        1698 non-null   int64
 13  Budget(INR)         1698 non-null   int64
dtypes: int64(3), str(11)
memory usage: 312.5 KB
None


#### Separating and cleaning each header of the new Cleaned Dataset

In [3]:
# Finding if rows deviate from standard 
deviants = df[~df['Release Period'].isin(['Normal', 'Holiday'])]
print("Deviants : ", len(deviants))    #0 : No deviations.

holiday_release = df["Release Period"].map({'Holiday': 1, 'Normal': 0}).to_frame(name="Holiday_Release")
print(holiday_release.info())

Deviants :  0
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Holiday_Release  1698 non-null   int64
dtypes: int64(1)
memory usage: 13.4 KB
None


In [4]:
# Finding if rows deviate from standard Yes / No
deviants = df[~df['Whether Remake'].isin(['Yes', 'No'])]
print("Deviants : ", len(deviants))    #0 : No deviations.

is_remake = df[["Whether Remake"]].replace({'Yes': 1, 'No': 0}).astype(int)
print(is_remake.info())

Deviants :  0
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Whether Remake  1698 non-null   int64
dtypes: int64(1)
memory usage: 13.4 KB
None


In [5]:
# Finding if rows deviate from standard Yes / No
deviants = df[~df['Whether Franchise'].isin(['Yes', 'No'])]
print("Deviants : ", len(deviants))    #0 : No deviations.

is_Franchise = df[["Whether Franchise"]].replace({'Yes': 1, 'No': 0}).astype(int)
print(is_Franchise.info())

Deviants :  0
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Whether Franchise  1698 non-null   int64
dtypes: int64(1)
memory usage: 13.4 KB
None


In [6]:
# Getting all genre cause we need separate binary variable for each since we cannot encode their discrete values using a single variable.   
# However, we will be skippng one of the genre to avoid thedummy variable trap, we need all the variables to be 
# linearly independent for regression to function properly.

genres = pd.get_dummies(df[["Genre"]], columns=["Genre"], drop_first=True, dtype=int)
print(genres.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Genre_adult         1698 non-null   int64
 1   Genre_animation     1698 non-null   int64
 2   Genre_comedy        1698 non-null   int64
 3   Genre_documentary   1698 non-null   int64
 4   Genre_drama         1698 non-null   int64
 5   Genre_fantasy       1698 non-null   int64
 6   Genre_horror        1698 non-null   int64
 7   Genre_love_story    1698 non-null   int64
 8   Genre_masala        1698 non-null   int64
 9   Genre_mythological  1698 non-null   int64
 10  Genre_rom_com       1698 non-null   int64
 11  Genre_suspense      1698 non-null   int64
 12  Genre_thriller      1698 non-null   int64
dtypes: int64(13)
memory usage: 172.6 KB
None


In [7]:
newcrew = df[["New Actor", "New Director", "New Music Director"]].replace({'Yes': 1, 'No': 0}).astype(int)
print(newcrew.info())

# We will be ignoring the names of actors and directors as it is extremetly difficult to encode their relative popularity
# accurately into variables, hence we arre only uses these 3 columns already provided in the dataset 
# and ignoring the next three columns.

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   New Actor           1698 non-null   int64
 1   New Director        1698 non-null   int64
 2   New Music Director  1698 non-null   int64
dtypes: int64(3)
memory usage: 39.9 KB
None


In [8]:
# We are using a normalized log scale for budget and screens to create a composite score that is then mapped to a 1-10 scale.

w_budget = 0.5  # Budget Weight : Assumed to be equal to screens weight, but will be adjusted later based on futher data analysis.
w_screens = 0.5 # Screens Weight

log_budget = np.log1p(df["Budget(INR)"])
log_screens = np.log1p(df["Number of Screens"])


norm_budget = (log_budget - log_budget.min()) / (log_budget.max() - log_budget.min())
norm_screens = (log_screens - log_screens.min()) / (log_screens.max() - log_screens.min())

composite = (w_budget * norm_budget) + (w_screens * norm_screens)

movie_scale = (1 + 9 * composite).round(3).to_frame(name="Movie_Scale")
print(movie_scale.info())


<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Movie_Scale  1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [9]:
rps = (df["Revenue(INR)"] / df["Number of Screens"]).to_frame(name="Revenue Per Screen").round(3)
print(rps.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Revenue Per Screen  1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [10]:
# our final point of interest : the Return On Interest

roi = (df["Revenue(INR)"] / df["Budget(INR)"]).to_frame(name="ROI").round(3)
print(roi.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ROI     1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [11]:
## Appending all the cleaned columns

df_clean = pd.concat([df[["Movie Name"]], is_remake, is_Franchise, holiday_release, genres, newcrew, df[["Budget(INR)", "Revenue(INR)", "Number of Screens"]], rps, movie_scale, roi], axis=1)
print(df_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Movie Name          1698 non-null   str    
 1   Whether Remake      1698 non-null   int64  
 2   Whether Franchise   1698 non-null   int64  
 3   Holiday_Release     1698 non-null   int64  
 4   Genre_adult         1698 non-null   int64  
 5   Genre_animation     1698 non-null   int64  
 6   Genre_comedy        1698 non-null   int64  
 7   Genre_documentary   1698 non-null   int64  
 8   Genre_drama         1698 non-null   int64  
 9   Genre_fantasy       1698 non-null   int64  
 10  Genre_horror        1698 non-null   int64  
 11  Genre_love_story    1698 non-null   int64  
 12  Genre_masala        1698 non-null   int64  
 13  Genre_mythological  1698 non-null   int64  
 14  Genre_rom_com       1698 non-null   int64  
 15  Genre_suspense      1698 non-null   int64  
 16  Genre_thriller   

In [12]:
print(df_clean.head(2))

   Movie Name  Whether Remake  Whether Franchise  Holiday_Release  \
0  Qaidi Band               0                  0                1   
1   Chaatwali               0                  0                1   

   Genre_adult  Genre_animation  Genre_comedy  Genre_documentary  Genre_drama  \
0            0                0             0                  0            1   
1            1                0             0                  0            0   

   Genre_fantasy  ...  Genre_thriller  New Actor  New Director  \
0              0  ...               0          1             0   
1              0  ...               0          1             1   

   New Music Director  Budget(INR)  Revenue(INR)  Number of Screens  \
0                   0    210000000       4500000                800   
1                   1      1000000       1075000                  1   

   Revenue Per Screen  Movie_Scale    ROI  
0              5625.0        7.803  0.021  
1           1075000.0        1.576  1.075  

[2

In [13]:
## Writing back to a new CSV file for further analysis and model building.
df_clean.to_csv('CleanedDataSet.csv', index=False)